# Connector Platform — Quickstart Notebook

**Prerequisites:**
1. Platform running: `CONNECTOR_DEV_MODE=1 cargo run --bin connector-platform`
2. This notebook: `pip install connector-sdk` then run cells top-to-bottom

No API key required in dev mode — any token works.

---

In [ ]:
# Step 0 — Install SDK (skip if already installed)
%pip install connector-sdk --quiet

In [ ]:
import os
from connector_sdk import ConnectorClient, ConnectorAgent

# In dev mode any token works. In prod: set CONNECTOR_API_KEY env var.
BASE_URL = os.getenv("CONNECTOR_BASE_URL", "http://localhost:8080")
API_KEY  = os.getenv("CONNECTOR_API_KEY",  "dev-token")

client = ConnectorClient(base_url=BASE_URL, api_key=API_KEY)

# Verify server is up
manifest = client._get("/api/v1")
print(f"✓ Connected to Connector Platform v{manifest.get('connector_platform')}")
print(f"  Mode: {manifest.get('mode')}")
print(f"  Tier: {manifest.get('license', {}).get('tier')}")

In [ ]:
# Step 1 — Register an agent
agent = ConnectorAgent(
    agent_pid="quickstart-agent",
    namespace="demo",
    base_url=BASE_URL,
    api_key=API_KEY,
)
result = agent.register(name="Quickstart Agent", description="Demo agent for notebook")
print(f"✓ Agent registered: pid={result.get('agent_pid')} status={result.get('status')}")

In [ ]:
# Step 2 — Write memory packets
facts = [
    "User prefers dark mode UI",
    "Project deadline is end of Q2 2026",
    "Tech stack: Rust backend, React frontend, PostgreSQL",
]

for fact in facts:
    r = agent.write_memory(fact, tags=["quickstart", "onboarding"])
    print(f"  ✓ Written CID={r.get('cid', 'n/a')[:16]}... content='{fact[:40]}'")

print(f"\n✓ {len(facts)} memory packets written to namespace='demo'")

In [ ]:
# Step 3 — Recall memory
# Use recall2 (canonical) for full packet fields
r = client._get("/api/v1/memory/recall2/demo", params={"limit": 10})
packets = r.get("packets", [])

print(f"✓ Recalled {len(packets)} packets from namespace='demo'\n")
for p in packets:
    ts = p.get('timestamp_ms', 0)
    import datetime
    ts_str = datetime.datetime.fromtimestamp(ts / 1000).strftime("%Y-%m-%d %H:%M") if ts else "n/a"
    print(f"  [{ts_str}] {p.get('content', '')[:60]}")

In [ ]:
# Step 4 — Verify a claim against grounding tables
r = client._post("/api/v1/safety/claims/verify", {
    "claim": "The project uses a React frontend",
    "agent_pid": "quickstart-agent",
    "namespace": "demo",
})
print(f"✓ Claim verification:")
print(f"  verdict:           {r.get('verdict', r.get('result', 'n/a'))}")
print(f"  hallucination_safe:{r.get('hallucination_safe', r.get('safe', 'n/a'))}")
print(f"  confidence:        {r.get('confidence', 'n/a')}")

In [ ]:
# Step 5 — Check trust score
r = client._get("/api/v1/monitor/trust")
print(f"✓ Platform trust score:")
print(f"  score:      {r.get('score', 'n/a')}")
print(f"  grade:      {r.get('grade', 'n/a')}")
print(f"  integrity:  {r.get('integrity', 'n/a')}")
print(f"  safety:     {r.get('safety', 'n/a')}")

In [ ]:
# Step 6 — Chat completion via AI Gateway
# Requires CONNECTOR_LLM_API_KEY or CONNECTOR_LLM_STUB=true
# The gateway lives at /v1/* (NOT /api/v1/v1/*)
import os, json, urllib.request

stub_mode = os.getenv("CONNECTOR_LLM_STUB", "false").lower() == "true"
llm_key   = os.getenv("CONNECTOR_LLM_API_KEY", "")

if not stub_mode and not llm_key:
    print("⚠ Skipping LLM call — set CONNECTOR_LLM_STUB=true or CONNECTOR_LLM_API_KEY")
    print("  Restart platform with: CONNECTOR_LLM_STUB=true cargo run --bin connector-platform")
else:
    payload = json.dumps({
        "model": "gpt-4o-mini",
        "messages": [{"role": "user", "content": "What is the project deadline?"}],
    }).encode()

    req = urllib.request.Request(
        f"{BASE_URL}/v1/chat/completions",
        data=payload,
        headers={"Content-Type": "application/json", "Authorization": f"Bearer {API_KEY}"},
        method="POST",
    )
    with urllib.request.urlopen(req) as resp:
        data = json.loads(resp.read())

    content = data["choices"][0]["message"]["content"]
    print(f"✓ AI Gateway response:")
    print(f"  {content[:200]}")

In [ ]:
# Step 7 — Run formal TLA+ invariant checks
r = client._get("/api/v1/safety/formal/verify")
invariants = r.get("invariants", [])
passed  = sum(1 for i in invariants if i.get("status") == "passed")
total   = len(invariants)
status  = "✓" if passed == total else "⚠"
print(f"{status} Formal invariants: {passed}/{total} passed")
for inv in invariants:
    icon = "✓" if inv.get("status") == "passed" else "✗"
    print(f"  {icon} {inv.get('name', inv.get('invariant', 'unknown'))}: {inv.get('status', 'n/a')}")

In [ ]:
# Step 8 — List running agents
r = client._get("/api/v1/agents")
agents = r.get("agents", [])
print(f"✓ {len(agents)} agent(s) registered:")
print(f"{'PID':<25} {'Status':<12} {'Namespace':<20} {'Tokens':>8}")
print("-" * 70)
for a in agents:
    print(f"{a.get('agent_pid',''):<25} {a.get('status',''):<12} {a.get('namespace',''):<20} {a.get('total_tokens_consumed',0):>8}")

In [ ]:
# Step 9 — Check billing usage
r = client._get("/api/v1/billing/usage")
print(f"✓ Billing usage:")
for k, v in r.items():
    if not isinstance(v, dict):
        print(f"  {k}: {v}")

---
## ✓ Quickstart complete!

You've just:
1. Connected to the Connector Platform
2. Registered an AI agent with namespace isolation
3. Written memory packets (CID-addressed, bi-temporal)
4. Recalled and displayed memory
5. Verified a claim for hallucination safety
6. Checked platform trust score
7. Made a chat completion via the OpenAI-compatible gateway
8. Run formal TLA+ invariant checks
9. Listed agents and billing usage

### Next steps

| Goal | Command / Endpoint |
|---|---|
| Deploy an agent manifest | `connector deploy run agent.yaml` |
| Run diagnostic checks | `connector doctor` |
| View memory tree | `connector memory tree quickstart-agent` |
| Trace LLM calls | `connector trace ls` |
| Run formal tests | `connector test all` |
| View API docs | http://localhost:8080/docs |
| Migration guide | http://localhost:8080/api/v1/docs/migration |
| Grafana dashboard | `GET /api/v1/monitor/grafana-dashboard` → import JSON |

Full docs: https://connector.dev/docs